# Day 34 PM – Part B: PCA Image Compression

Simple PCA-based image compression demo using a generated RGB image loaded with matplotlib.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.metrics import mean_squared_error


In [ ]:
x = np.linspace(0, 1, 128)
y = np.linspace(0, 1, 128)
xx, yy = np.meshgrid(x, y)
img = np.zeros((128, 128, 3))
img[:, :, 0] = xx
img[:, :, 1] = yy
img[:, :, 2] = 0.5 * np.sin(6 * np.pi * xx) * np.cos(6 * np.pi * yy) + 0.5
plt.imsave('sample_pca_image.png', np.clip(img, 0, 1))
loaded = plt.imread('sample_pca_image.png')[:, :, :3]
print('Loaded image shape:', loaded.shape)


In [ ]:
components_list = [5, 20, 50, 100]
reconstructed_images = []
rows = []

for n_comp in components_list:
    channels = []
    for c in range(3):
        channel = loaded[:, :, c]
        pca = PCA(n_components=n_comp)
        transformed = pca.fit_transform(channel)
        reconstructed = pca.inverse_transform(transformed)
        channels.append(reconstructed)
    recon_img = np.stack(channels, axis=2)
    recon_img = np.clip(recon_img, 0, 1)
    reconstructed_images.append((n_comp, recon_img))
    original_size = loaded.size
    compressed_size = loaded.shape[1] * n_comp * 3 + n_comp * 3 + loaded.shape[1] * 3
    ratio = original_size / compressed_size
    mse = mean_squared_error(loaded.reshape(-1, 3), recon_img.reshape(-1, 3))
    rows.append({'n_components': n_comp, 'compression_ratio': ratio, 'mse': mse})

metrics_df = pd.DataFrame(rows)
print(metrics_df.round(4).to_string(index=False))


In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(18, 4))
axes[0].imshow(loaded)
axes[0].set_title('Original')
axes[0].axis('off')

for ax, (n_comp, recon_img) in zip(axes[1:], reconstructed_images):
    ax.imshow(recon_img)
    ax.set_title(f'PCA {n_comp}')
    ax.axis('off')

plt.tight_layout()
plt.savefig('pca_compression_grid.png', dpi=150)
plt.show()
